# N19 · Train-Infer Mismatch：K3 KL 在 RL 训练里到底意味着什么

> 配套 `labs/l29.5_train_infer_mismatch/`。
>
> 读完跑完之后你应该能解释：
> 1. K3 KL 怎么算？为什么不用 K1？
> 2. TIS 截断 vs MIS mask 在数学上的区别
> 3. 训练中段 K3 KL 上升而 reward 还稳定时，怎么决定干预
>
> 参考：[slime mismatch 博客](../github_repo/Awesome-ML-SYS-Tutorial/rlhf/slime/mismatch/blog-cn.md)、[Schulman: Approximating KL](http://joschu.net/blog/kl-approx.html)


## 1. K3 KL 估计器

$$ k_3(x) = \frac{p(x)}{q(x)} - 1 - \log\frac{p(x)}{q(x)} $$

性质：
- $k_3 \geq 0$，且 $p = q$ 时严格为 0；
- 是 $\mathrm{KL}(p\|q)$ 的**低方差无偏估计器**；
- 在 RL 监控里：$q$ 用 rollout engine 的 logp，$p$ 用 train engine 的 logp。

相比之下 $k_1 = \log(p/q)$ 虽然也是无偏的，但单样本会出现负值（对监控不友好），且方差大。


In [ ]:
import torch

def k1(logp, logq):
    return (logp - logq).mean()

def k3(logp, logq):
    log_ratio = logp - logq
    return (torch.exp(log_ratio) - 1.0 - log_ratio).mean()

torch.manual_seed(0)
log_p = torch.randn(1024) * 0.1 - 1.0

print(f"{'scenario':30s}  {'k1':>10s}  {'k3':>10s}")
print(f"{'aligned (p=q)':30s}  {k1(log_p, log_p).item():10.5f}  {k3(log_p, log_p).item():10.5f}")

log_q1 = log_p + torch.randn_like(log_p) * 0.01
print(f"{'micro mismatch (sigma=0.01)':30s}  {k1(log_p, log_q1).item():10.5f}  {k3(log_p, log_q1).item():10.5f}")

log_q2 = log_p + torch.randn_like(log_p) * 0.5
print(f"{'large mismatch (sigma=0.5)':30s}  {k1(log_p, log_q2).item():10.5f}  {k3(log_p, log_q2).item():10.5f}")

log_q3 = log_p + torch.randn_like(log_p) * 2.0
print(f"{'huge mismatch (sigma=2.0)':30s}  {k1(log_p, log_q3).item():10.5f}  {k3(log_p, log_q3).item():10.5f}")

**观察**：
- aligned 情况：k1, k3 都接近 0
- micro mismatch：k1 可能是个微小正/负数，k3 始终为很小的正数
- large mismatch：两者都增加，但 k3 增长更陡（指数项放大）

这就是为什么 RL 框架监控用 k3：**它对早期细微 drift 不敏感，对真正失控的 mismatch 敏感**——天然的 alarm 触发器。


## 2. TIS 截断 vs MIS mask

两者都用 `ratio = exp(logp_new - logp_old)`：
- **TIS**：`ratio.clamp(lo, hi)` —— 越界的 token 拉回边界，仍参与训练
- **MIS**：越界的 token **mask 为 0**，不参与梯度

哪种更好？分场景：
- 训练前期 mismatch 小，TIS 足够
- 训练后期 ratio 出现 100+ 的 outlier 时，必须 MIS 把这些 token 直接 drop


In [ ]:
import torch

log_old = torch.tensor([-1.0, -2.0, -5.0, -1.0, -1.0])
log_new = torch.tensor([-1.0, -1.5, 0.0, -1.0, -2.5])
ratio = torch.exp(log_new - log_old)

lo, hi = 0.5, 2.0
tis = ratio.clamp(lo, hi)
mask = ((ratio >= lo) & (ratio <= hi)).float()
mis = ratio * mask

print(f"{'token':>6s}  {'ratio':>10s}  {'TIS':>10s}  {'MIS':>10s}")
for i in range(len(ratio)):
    print(f"{i:>6d}  {ratio[i].item():10.4f}  {tis[i].item():10.4f}  {mis[i].item():10.4f}")

print(f"\nTIS sum: {tis.sum().item():.4f} (越界 token 贡献了 hi=2.0)")
print(f"MIS sum: {mis.sum().item():.4f} (越界 token 贡献了 0)")

## 3. 何时干预？K3 上升 + reward 稳定的解读

真实事故（Qwen30B-A3B）：
- 320 步：grad norm 从 0.07 跌到 0.02（前兆）
- ~330 步：reward 骤降，K3 KL 飙升
- 340 步：reward 暂时恢复，但 grad norm 已经不正常

**判断准则**：
1. K3 < 1e-4 ：dense 模型正常，无需干预
2. K3 在 1e-4 ~ 1e-3 持续上升：开 `--use-tis` 加保险
3. K3 > 1e-2 且 grad norm 异常：必须开 `--use-mis` + `--use-rs` + veto
4. K3 > 1e-1：MoE 模型典型崩溃前夜，立刻 stop & roll back

## 4. 自检 / 面试题

1. K1 是无偏估计器，为什么 RL 监控宁愿用 K3？（答：K3 始终非负，且方差更小，便于 alarm）
2. MoE 模型的 K3 KL 通常比 dense 模型大 1–2 个数量级，原因是什么？（答：路由不一致 → 激活 expert 不同 → logits 差异被放大）
3. 启用 TIS / MIS 会不会损害正常训练性能？（答：在 mismatch 小的情况下不会，slime 实验里 4 种配置 reward 趋同；mismatch 大时 MIS 能从崩溃边缘救回）
4. Veto 阈值为什么用对数比较而不是 `exp(logp) < threshold`？（答：避免 exp 下溢，对 logp 直接比较 `>= log(threshold)` 是数值稳定的）
